<img src="https://raw.githubusercontent.com/ralf-42/Agenten/main/07_image/banner_ki_agenten.png" class="logo" width="700"/>

<p><font size="5" color='grey'> <b> Chat & Memory </b></font> </br></p>

---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/GenAI.git#subdirectory=04_modul
!uv pip install --system -q langgraph langchain_openai
from genai_lib.utilities import check_environment, get_ipinfo, setup_api_keys, mprint, install_packages
setup_api_keys(['OPENAI_API_KEY', 'HF_TOKEN'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()


# 4 | Long-term Memory - Ausblick
---

**Problem:** Short-term Memory ist chronologisch und begrenzt. Es eignet sich nicht für das Speichern von großen, permanenten Wissensbasen oder Fakten.

**Lösung:** LangGraph **Store** (Long-term Memory).

Der Store speichert Fakten in **Namespaces** (typischerweise `user_id` oder `topic`) und ermöglicht die **semantische Suche** (Vektorsuche), wodurch nur relevante Fakten in den Prompt geladen werden (RAG-Prinzip).

<p><font color='black' size="5">
Die zwei Memory-Typen in LangGraph
</font></p>

Im LangChain/LangGraph-Umfeld wird das Gedächtnis klar getrennt, um Multi-User-Systeme zu ermöglichen:

| Aspekt | Short-term (Checkpointer) | Long-term (Store) |
|--------|---------------------------|-------------------|
| **Zweck** | Gesprächskontext pro Thread | Wissensbasis über Threads hinweg |
| **Scope** | **`thread_id`** (Session) | **`user_id`** / Namespace (Global) |
| **Speicher** | Liste von Nachrichten (chronologisch) | Fakten / Präferenzen (semantisch) |
| **Herausforderung** | Längenbegrenzung (Trimming) | Relevante Suche (RAG) |
| **Technologie** | `InMemorySaver`, `SqliteSaver` | `InMemoryStore` mit Index, `PostgresStore` |


In [ ]:
# Importe
from langgraph.store.memory import InMemoryStore
from langchain_core.runnables import RunnableConfig

<p><font color='black' size="5">
Pre-Processing: Datensammlung
</font></p>

In [ ]:
# Store mit OpenAI Embeddings (1536 Dimensionen für text-embedding-3-small)
# 🛑 WICHTIG: Der Index ermöglicht die semantische (Ähnlichkeits-)Suche!
store = InMemoryStore(index={
    "dims": 1536,
    "embed": "openai:text-embedding-3-small"
})

# Fakten zu User "Max" speichern (Namespace: user/max)
USER_ID_MAX = "max_42"
user_ns = ("user", USER_ID_MAX)

# Speichere Schlüssel-Wert-Paare im Store (könnten auch aus Summarization kommen)
store.put(user_ns, "proj_2020", {"text": "Max hat 2020 ein Chatbot-Projekt gemacht"})
store.put(user_ns, "lang_python", {"text": "Maxs Lieblingssprache ist Python"})
store.put(user_ns, "city_koeln", {"text": "Max wohnt in Köln"})

print(f"✅ Long-term Store mit {len(store.search(user_ns, limit=10))} Einträgen für User '{USER_ID_MAX}' erstellt")

In [ ]:
# Semantische Suche (nicht nur String-Matching!)
def retrieve_memory_semantic(query: str, user_id: str, k: int = 2):
    """Sucht relevante Erinnerungen per semantischer Vektorsuche im Store."""
    ns = ("user", user_id)
    # 🛑 Semantische Suche mit 'query' im optionalen zweiten Argument
    results = store.search(ns, query=query, limit=k)

    if not results:
        return "(Keine relevanten Erinnerungen gefunden)"

    return "\n".join([item.value["text"] for item in results])

# Test der Suche
print("🔍 Semantische Suche: 'Welche Projekte?'")
print(retrieve_memory_semantic("Welche Projekte hat er gemacht?", USER_ID_MAX))
print("\n🔍 Semantische Suche: 'Programmiersprachen' (trotzdem Treffer bei 'Lieblingssprache')")
print(retrieve_memory_semantic("Programmiersprachen", USER_ID_MAX))

<p><font color='black' size="5">
Workflow
</font></p>

In [ ]:
# Chat mit Store-Integration
def chat_node_with_store(state: MessagesState, config: RunnableConfig):
    """Nutzt Long-term Store (Wissensbasis) zusätzlich zur Short-term Historie."""

    # 🛑 Wichtig: user_id wird als Namespace für den Store genutzt
    user_id = config["configurable"]["user_id"]
    user_query = state["messages"][-1].content

    # 1. Semantische Suche in Store (RAG-Schritt)
    memories = retrieve_memory_semantic(user_query, user_id, k=2)

    # 2. Relevante Fakten als System-Kontext hinzufügen
    messages = [
        SystemMessage(content=f"{system_prompt}\n\nRelevante Informationen aus Wissensbasis:\n{memories}")
    ] + state["messages"]

    response = llm.invoke(messages)
    return {"messages": [response]}


<p><font color='black' size="5">
Graph
</font></p>

In [ ]:
# Graph mit Store erstellen
workflow_store = StateGraph(state_schema=MessagesState)
workflow_store.add_node("chat", chat_node_with_store)
workflow_store.add_edge(START, "chat")
workflow_store.add_edge("chat", END)

# Short-term Checkpointer (MemorySaver) bleibt erhalten
graph_store = workflow_store.compile(checkpointer=MemorySaver())
display(Image(graph.get_graph().draw_mermaid_png()))

<p><font color='black' size="5">
Ausführung
</font></p>

In [ ]:
def chat_with_store(thread_id: str, user_id: str, user_input: str):
    """Chattet mit dem Bot unter Verwendung von Store (semantische Suche)."""

    config = {
        "configurable": {
            "thread_id": thread_id, # Short-term (Historie)
            "user_id": user_id      # Long-term (Store-Namespace)
        }
    }
    input_message = {"messages": [HumanMessage(content=user_input)]}

    result = graph_store.invoke(input_message, config=config)

    print(f"🧑‍🦱 Mensch: {user_input}")
    print(f"🤖 KI: {result['messages'][-1].content}\n")

    return result['messages'][-1].content

# Beispiel: Die KI kann auf Store-Wissensbasis zugreifen (semantische Suche!)
chat_with_store("store_max_t1", USER_ID_MAX, "Wann hat Max sein Chatbot-Projekt gemacht?")
chat_with_store("store_max_t1", USER_ID_MAX, "Welche Programmiersprache mag Max?")

# A | Aufgaben
---


Die Aufgabenstellungen unten bieten Anregungen. Sie können aber auch gerne eigene Aufgaben verwenden.



<p><font color='black' size="5">
Aufgabe 1: Multi-User Chatbot mit LangGraph (Short-term)
</font></p>

**Schwierigkeit:** ⭐⭐

Erstellen Sie einen Chatbot unter Verwendung von **Abschnitt 2** (`graph`), der:
- Mindestens 3 verschiedene User-Threads verwaltet (z.B. "max_session", "emma_session", "ralf_session").
- Jeden User beim Namen kennt.
- Beim Thread-Wechsel den Kontext des jeweiligen Users korrekt behält (zeigen Sie dies, indem Sie zwischen zwei Threads hin- und herwechseln).


<p><font color='black' size="5">
Aufgabe 2: Memory mit Trimming-Strategie
</font></p>

**Schwierigkeit:** ⭐⭐⭐

Implementieren Sie einen Chatbot unter Verwendung von **Abschnitt 3.2** (`graph_summary`), der:
1. **`max_messages_before_trim`** auf einen kleinen Wert (z.B. 5) setzt.
2. Eine längere Konversation (z.B. 7 Schritte) durchführt, sodass die **Zusammenfassungslogik** ausgelöst wird (achten Sie auf den `⚠️` Hinweis).
3. Prüfen Sie mit `show_thread_history(thread_id)`, ob der Checkpointer immer noch **alle 7 Nachrichten** speichert, aber der Chat-Node **nur die relevanten Nachrichten** verwendet.

*(Hinweis: Der Checkpointer speichert immer die volle Historie; die Management-Strategie entscheidet, was dem LLM präsentiert wird.)*